## Imports

In [96]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

## Utils

In [97]:
def get_data_root():
  '''
    Ritorna il percorso della cartella contenente i dati in base all'ambiente di esecuzione.
  '''
  try:
      import google.colab
      from google.colab import drive

      try:
          drive.mount("/content/drive", force_remount=True)
          return "/content/drive/MyDrive/ColabContent/Data_analytics"
      except Exception:
          print("Drive non montabile")
          return "/content"

  except ImportError:
      return "../../data"

print(get_data_root())

../../data


## Global variables

In [98]:
DATA_ROOT = get_data_root()
DATASET_PATH = f"{DATA_ROOT}/Dataset2526/train.csv"
TRAIN_SET_PATH = f"{DATA_ROOT}/train_processed.csv"
VAL_SET_PATH = f"{DATA_ROOT}/val_processed.csv"
TEST_SET_PATH = f"{DATA_ROOT}/test_processed.csv"
SEED = 42

# Modeling (tradition ML models)

In [99]:
train = pd.read_csv(TRAIN_SET_PATH)
val = pd.read_csv(VAL_SET_PATH)
test = pd.read_csv(TEST_SET_PATH)

X_train = train.drop(columns=['grade'])
y_train = train["grade"]

X_val = val.drop(columns=['grade'])
y_val = val["grade"]

X_test = test.drop(columns=['grade'])
y_test = test["grade"]

print(f"TRAIN:\nX: {X_train.shape}\ny: {y_train.shape}")
print(f"VAL:\nX: {X_val.shape}\ny: {y_val.shape}")
print(f"TEST:\nX: {X_test.shape}\ny: {y_test.shape}")

TRAIN:
X: (33747, 98)
y: (33747,)
VAL:
X: (14830, 98)
y: (14830,)
TEST:
X: (14831, 98)
y: (14831,)


## 1. KNN

### Train knn

In [100]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('knn', KNeighborsClassifier())
])

param_grid = {
    'pca__n_components': [20, 30, 40, 50, 60, 70, 80, 90],
    'knn__n_neighbors': [31, 51, 101, 201, 401, 801] 
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print(f"Miglior score: {grid_search.best_score_}")
print(f"Migliori parametri: {grid_search.best_params_}")

Fitting 5 folds for each of 48 candidates, totalling 240 fits
Miglior score: 0.399413136651246
Migliori parametri: {'knn__n_neighbors': 401, 'pca__n_components': 80}


### Test knn

In [101]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_val)


print(f"Accuracy: {accuracy_score(y_val, y_pred)}")
print(classification_report(y_val, y_pred))

Accuracy: 0.4198246797033041
              precision    recall  f1-score   support

           0       0.51      0.81      0.63      2603
           1       0.46      0.44      0.45      3806
           2       0.46      0.30      0.36      3753
           3       0.34      0.25      0.29      2073
           4       0.23      0.21      0.22      1206
           5       0.24      0.45      0.31       762
           6       0.33      0.36      0.35       627

    accuracy                           0.42     14830
   macro avg       0.37      0.40      0.37     14830
weighted avg       0.42      0.42      0.41     14830

